# 03 — Model Training
**SF Crime Classification | Step 3 of 4**

> Prerequisite: Run `02_feature_engineering.ipynb` first.

In [ ]:
%pip install xgboost lightgbm -q

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, log_loss, f1_score,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import lightgbm as lgb

CATALOG     = "workspace"
SCHEMA      = "default"
VOLUME_PATH = f"/Volumes/workspace/default/sf_crime_data"
print("✅ Libraries loaded")

In [ ]:
FEATURES = [
    "Hour","Minute","Month","Year","DayOfMonth","DayOfYear","WeekOfYear","Quarter","Season",
    "IsWeekend","IsNight","IsLateNight","IsMorning","IsAfternoon","IsEvening","IsRushHour","IsBusinessHours",
    "HourSin","HourCos","MonthSin","MonthCos","DowSin","DowCos","DaySin","DayCos",
    "X","Y","DistCenter","CoordFreq","X_round2","Y_round2","X_round3","Y_round3","GridX","GridY",
    "IsBlock","IsIntersection","StreetNum","StreetNumBin",
    "District_enc","DayOfWeek_enc","DistrictRate",
    "District_Hour","District_DayOfWeek","Hour_DayOfWeek","District_IsNight","District_IsWeekend",
]

pdf = spark.read.table(f"{CATALOG}.{SCHEMA}.features_train").toPandas()
X   = pdf[FEATURES].values
y   = pdf["Target"].values.astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
n_classes = int(y.max()) + 1
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Classes: {n_classes}")

In [ ]:
def compute_metrics(name, y_true, y_pred, y_prob):
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    m = {
        "accuracy":          accuracy_score(y_true, y_pred),
        "log_loss":          log_loss(y_true, y_prob),
        "f1_weighted":       f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro":          f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc":               matthews_corrcoef(y_true, y_pred),
        "roc_auc_macro":     roc_auc_score(y_bin, y_prob, multi_class="ovr", average="macro"),
        "roc_auc_weighted":  roc_auc_score(y_bin, y_prob, multi_class="ovr", average="weighted"),
    }
    print(f"\n{name}")
    for k, v in m.items():
        print(f"  {k:<22}: {v:.4f}")
    return m

In [ ]:
print("Training XGBoost...")
model_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric="mlogloss", random_state=42, n_jobs=-1, verbosity=0)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
metrics_xgb = compute_metrics("XGBoost", y_val, model_xgb.predict(X_val), model_xgb.predict_proba(X_val))

In [ ]:
print("Training LightGBM...")
model_lgbm = lgb.LGBMClassifier(
    n_estimators=500, max_depth=10, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1)
model_lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])
metrics_lgbm = compute_metrics("LightGBM", y_val, model_lgbm.predict(X_val), model_lgbm.predict_proba(X_val))

In [ ]:
# ── Compare & save best model to Volume ───────────────────
results = {
    "XGBoost":  (model_xgb,  metrics_xgb),
    "LightGBM": (model_lgbm, metrics_lgbm),
}
best_name, (best_model, best_metrics) = max(
    results.items(), key=lambda x: x[1][1]["roc_auc_weighted"])

print(f"\n🏆 Best model : {best_name}")
print(f"   ROC-AUC    : {best_metrics['roc_auc_weighted']:.4f}")
print(f"   Accuracy   : {best_metrics['accuracy']:.4f}")

model_path = f"{VOLUME_PATH}/best_model.joblib"
joblib.dump({"model": best_model, "name": best_name, "metrics": best_metrics}, model_path)
print(f"\n✅ Saved to: {model_path}")

In [ ]:
# ── Side-by-side comparison ───────────────────────────────
cmp = pd.DataFrame([
    {"Model": "XGBoost",  **metrics_xgb},
    {"Model": "LightGBM", **metrics_lgbm},
]).set_index("Model").round(4)
print(cmp.to_string())

---
**Next:** Run `04_inference_submission.ipynb`.